# Switzerland relief map — ILLgraben

This notebook draws a clipped shaded-relief map of Switzerland with city labels, the ILLgraben study site, and coordinate crosshairs.

Run the cells from top to bottom. The final cell contains all editable visual settings and produces exactly one inline figure, plus PNG and PDF files.

## Environment note

Use the `pygmt` Conda environment as the notebook kernel. If PyGMT reports an incompatible GMT version, run the following in **Anaconda Prompt**, then restart Jupyter/PyCharm:

```powershell
conda install -n pygmt -c conda-forge "gmt>=6.5" pygmt libgdal-jp2openjpeg ipykernel
conda run -n pygmt python -m ipykernel install --user --name pygmt --display-name "Python (pygmt)"
```


In [ ]:
# Imports and paths
import os
from pathlib import Path

os.environ["GDAL_DRIVER_PATH"] = (
    r"D:\Anaconda\envs\pygmt\Library\lib\gdalplugins"
)

import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patheffects as pe
from matplotlib.colors import LightSource, LinearSegmentedColormap
from IPython.display import display
from scipy.ndimage import distance_transform_edt
from shapely import contains_xy

import pygmt
import cartopy.crs as ccrs
import cartopy.io.shapereader as shpreader

output_dir = Path(
    r"D:\Pycharm_Project\Find_precursor_signs\ILLgarben"
    r"\uilt_oneday_all\uilt_paper_figure\map"
)
output_dir.mkdir(parents=True, exist_ok=True)

In [ ]:
# Map data and annotations
region = [5.65, 10.65, 45.65, 47.90]  # west, east, south, north

site_lon = 7 + 35 / 60
site_lat = 46 + 25 / 60

cities = {
    "Geneva": (6.1432, 46.2044),
    "Lausanne": (6.6323, 46.5197),
    "Bern": (7.4474, 46.9480),
    "Zürich": (8.5417, 47.3769),
    "Chur": (9.5300, 46.8508),
}

city_labels = {
    "Geneva": (5.88, 46.02),
    "Lausanne": (6.30, 46.68),
    "Bern": (7.10, 47.08),
    "Zürich": (8.52, 47.58),
    "Chur": (9.35, 47.02),
}

In [ ]:
# Switzerland boundary
shp_path = shpreader.natural_earth(
    resolution="10m",
    category="cultural",
    name="admin_0_countries",
)

switzerland = None
for record in shpreader.Reader(shp_path).records():
    attrs = record.attributes
    if (
        attrs.get("ADM0_A3") == "CHE"
        or attrs.get("SOV_A3") == "CHE"
        or attrs.get("ISO_A3") == "CHE"
    ):
        switzerland = record.geometry
        break

if switzerland is None:
    raise RuntimeError("Switzerland was not found in the Natural Earth dataset.")

In [ ]:
# DEM reading and preprocessing
dem = pygmt.datasets.load_earth_relief(
    resolution="15s",
    region=region,
)

lon = np.asarray(dem.coords["lon"].values)
lat = np.asarray(dem.coords["lat"].values)
elevation = np.asarray(dem.values, dtype=float)

if lat[0] > lat[-1]:
    lat = lat[::-1]
    elevation = elevation[::-1, :]

if lon[0] > lon[-1]:
    lon = lon[::-1]
    elevation = elevation[:, ::-1]

xx, yy = np.meshgrid(lon, lat)
inside = contains_xy(switzerland, xx, yy)

nearest_indices = distance_transform_edt(
    ~inside,
    return_distances=False,
    return_indices=True,
)
elevation_filled = elevation[tuple(nearest_indices)]

mean_lat = np.mean(lat)
dx = abs(np.mean(np.diff(lon))) * 111_320 * np.cos(np.deg2rad(mean_lat))
dy = abs(np.mean(np.diff(lat))) * 110_540
valid_values = elevation[inside]

In [ ]:
# Plot settings
figsize = (7.2, 4.3); png_dpi = 600; output_stem = "switzerland_relief_map"
font_family = "Arial"; label_fs = 18; font_weight = "bold"
background = "#FFFFFF"; 
relief_colors = ["#999999", "#C8C8C8", "#F2F2F2"]
# relief_colors = ["#888888", "#BBBBBB", "#F5F5F5"]
# relief_colors = [
#     "#5B8C5A",
#     "#A8B96E",
#     "#D1B477",
#     "#9B7653",
#     "#FFFFFF",
# ]




boundary_color = "#666666"; crossline_color = "#222222"; city_color = "#000000"
site_color = "#E31A1C"; text_color = "#000000"; text_outline = "#FFFFFF"
boundary_lw = 0.55; crossline_lw = 1.6; site_edge_lw = 1.60
city_size = 54; site_size = 72; text_outline_lw = 2.8
azimuth = 315; altitude = 42; vertical_exaggeration = 2.0
low_percentile = 2; high_percentile = 98; interpolation = "bilinear"
site_label_dx = 0.30; site_label_dy = 0.15
longitude_label_y = region[2] + 0.02
latitude_label_x = region[1] - 0.08; latitude_label_dy = -0.12
z_relief = 1; z_boundary = 2; z_crossline = 3; z_city = 5; z_city_text = 6; z_site = 7

plt.rcParams.update({
    "font.family": font_family,
    "font.sans-serif": [font_family],
    "font.size": label_fs,
    "pdf.fonttype": 42,
    "ps.fonttype": 42,
})

# Plot
relief_cmap = LinearSegmentedColormap.from_list("switzerland_relief", relief_colors)
light = LightSource(azdeg=azimuth, altdeg=altitude)
relief = light.shade(
    elevation_filled,
    cmap=relief_cmap,
    vert_exag=vertical_exaggeration,
    dx=dx,
    dy=dy,
    vmin=np.percentile(valid_values, low_percentile),
    vmax=np.percentile(valid_values, high_percentile),
    blend_mode="soft",
)
relief = np.array(relief, copy=True)
relief[..., 3] = inside.astype(float)

data_crs = ccrs.PlateCarree()
map_crs = ccrs.UTM(zone=32)
fig = plt.figure(figsize=figsize, facecolor=background)
ax = fig.add_subplot(1, 1, 1, projection=map_crs)
ax.set_extent(region, crs=data_crs)
ax.set_facecolor(background)

ax.imshow(
    relief,
    extent=[lon.min(), lon.max(), lat.min(), lat.max()],
    origin="lower",
    transform=data_crs,
    interpolation=interpolation,
    zorder=z_relief,
)
ax.add_geometries(
    [switzerland],
    crs=data_crs,
    facecolor="none",
    edgecolor=boundary_color,
    linewidth=boundary_lw,
    zorder=z_boundary,
)
# ax.plot(
#     [site_lon, site_lon],
#     [region[2] + 0.08, region[3] - 0.05],
#     color=crossline_color,
#     linewidth=crossline_lw,
#     transform=data_crs,
#     zorder=z_crossline,
# )
# ax.plot(
#     [region[0] + 0.08, region[1] - 0.08],
#     [site_lat, site_lat],
#     color=crossline_color,
#     linewidth=crossline_lw,
#     transform=data_crs,
#     zorder=z_crossline,
# )
# 将研究点经纬度转换为地图投影坐标
site_x, site_y = map_crs.transform_point(
    site_lon,
    site_lat,
    data_crs,
)

# 严格垂直线
ax.axvline(
    x=site_x,
    ymin=0.02,
    ymax=0.98,
    color=crossline_color,
    linewidth=crossline_lw,
    zorder=z_crossline,
)

# 严格水平线
ax.axhline(
    y=site_y,
    xmin=0.02,
    xmax=0.98,
    color=crossline_color,
    linewidth=crossline_lw,
    zorder=z_crossline,
)



for x, y in cities.values():
    ax.scatter(
        x,
        y,
        s=city_size,
        color=city_color,
        edgecolor=city_color,
        linewidth=site_edge_lw,
        transform=data_crs,
        zorder=z_city,
    )

text_effect = [
    pe.withStroke(
        linewidth=text_outline_lw,
        foreground=text_outline,
        alpha=0.9,
    )
]

for name, (tx, ty) in city_labels.items():
    ax.text(
        tx,
        ty,
        name,
        transform=data_crs,
        ha="center",
        va="center",
        fontsize=label_fs,
        fontfamily=font_family,
        fontweight=font_weight,
        color=text_color,
        path_effects=text_effect,
        zorder=z_city_text,
    )

ax.scatter(
    site_lon,
    site_lat,
    s=site_size,
    color=site_color,
    edgecolor=background,
    linewidth=site_edge_lw,
    transform=data_crs,
    zorder=z_site,
)
ax.text(
    site_lon + site_label_dx,
    site_lat + site_label_dy,
    "ILLgraben",
    transform=data_crs,
    ha="center",
    va="center",
    fontsize=label_fs,
    fontfamily=font_family,
    fontweight=font_weight,
    color=text_color,
    path_effects=text_effect,
    zorder=z_site,
)
ax.text(
    site_lon,
    longitude_label_y,
    "7°37′ E",
    transform=data_crs,
    ha="center",
    va="bottom",
    fontsize=label_fs,
    fontfamily=font_family,
    fontweight=font_weight,
    color=text_color,
    clip_on=False,
    zorder=z_site,
)
ax.text(
    latitude_label_x,
    site_lat + latitude_label_dy,
    "46°17′ N",
    transform=data_crs,
    ha="right",
    va="top",
    fontsize=label_fs,
    fontfamily=font_family,
    fontweight=font_weight,
    color=text_color,
    clip_on=False,
    zorder=z_site,
)

# Axis and layout
ax.set_axis_off()

# Save and display once
fig.savefig(
    output_dir / f"{output_stem}.png",
    dpi=png_dpi,
    bbox_inches="tight",
    pad_inches=0.03,
    facecolor=background,
)
fig.savefig(
    output_dir / f"{output_stem}.pdf",
    bbox_inches="tight",
    pad_inches=0.03,
    facecolor=background,
)
display(fig)
plt.close(fig)

## Color editing

Edit `relief_colors` at the top of the final plotting cell. Colors are ordered from low to high elevation.

Examples:

```python
# Grayscale
relief_colors = ["#999999", "#C8C8C8", "#F2F2F2"]

# Natural terrain
relief_colors = ["#6F9B60", "#C6B578", "#9A7655", "#EEEEEE", "#FFFFFF"]

# Light journal style
relief_colors = ["#D8DFC1", "#D2C59C", "#B09A78", "#EEEAE2"]
```

The other colors are controlled by `boundary_color`, `crossline_color`, `city_color`, `site_color`, `text_color`, and `background` in the same cell.
